# Action-Rate Threshold Backtesting (Headline Number)

This notebook computes the headline number by backtesting a per-identity action-rate threshold against HuggingFace's incident data.
The goal is to determine the hours before the Day-3 main-campaign spike (Hour 48).
*Note: Real detection was later than Hour 48, so this metric serves as a conservative lower bound.*

### Threshold Justification
The threshold of 1,000 actions/identity/day was selected because it is orders of magnitude above plausible legitimate automated use for this infrastructure.

### Assumptions:
1. **Uniform Action Distribution**: The 3,779 actions on Day 1 were uniformly distributed over the 24-hour period.
2. **Single Identity Attribution**: All Day-1 actions are attributable to a single compromised identity (the public record does not break counts down by identity).


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the public record data
with open('data/incident_public_record.json', 'r') as f:
    data = json.load(f)

# Extract daily counts
days = data['days']
df = pd.DataFrame(days)
df['date'] = pd.to_datetime(df['date'])
df['cumulative_actions'] = df['actions'].cumsum()

# Day 1 actions
day1_actions = df.iloc[0]['actions']
print(f"Day 1 actions: {day1_actions}")

# Sensitivity Band
thresholds = [500, 1000, 2000]
print("\nSensitivity Band Analysis:")
for t in thresholds:
    hours_to_trigger = (t / day1_actions) * 24
    hours_earlier = 48 - hours_to_trigger
    print(f"Threshold {t:4d} -> {hours_earlier:.1f} hours before the Day-3 main-campaign spike (Hour 48)")

headline_trigger = (1000 / day1_actions) * 24
headline_earlier = 48 - headline_trigger
print(f"\nHeadline Metric: {headline_earlier:.1f} hours before the Day-3 main-campaign spike (using 1,000 threshold).")

# Plot the cumulative actions and thresholds
plt.figure(figsize=(10, 5))
plt.plot(range(len(df)), df['cumulative_actions'], marker='o', label='Cumulative Actions')
for t in thresholds:
    plt.axhline(y=t, linestyle='--', label=f'Threshold ({t})')
plt.title('Action Rate Backtesting vs 2026-07 Incident')
plt.xlabel('Days since incident start')
plt.ylabel('Cumulative Actions')
plt.xticks(range(len(df)), df['date'].dt.strftime('%Y-%m-%d'))
plt.legend()
plt.grid(True)
plt.savefig('../report/figures/action_rate_threshold.png')
print("Saved plot to report/figures/action_rate_threshold.png")
